In [125]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
from scipy.stats import chi2_contingency

In [126]:
df = pd.read_csv('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Data/Fraud_Data.csv')
for i in df.columns:
    print(i)

transaction_id
is_fraud
transaction_time
transaction_amount
card_network
card_type
purchaser_email_domain
device_type
is_identity_seen_before
user_os
user_browser
environment
environment_freq
environment_risk
time_hour
time_day
time_diff
time_diff_min
amt_log
amt_bins
new_day
day_id
day_of_week
time_diff_log


In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           590540 non-null  int64  
 1   is_fraud                 590540 non-null  int64  
 2   transaction_time         590540 non-null  int64  
 3   transaction_amount       590540 non-null  float64
 4   card_network             588963 non-null  object 
 5   card_type                588969 non-null  object 
 6   purchaser_email_domain   496084 non-null  object 
 7   device_type              140810 non-null  object 
 8   is_identity_seen_before  129340 non-null  object 
 9   user_os                  144233 non-null  object 
 10  user_browser             144233 non-null  object 
 11  environment              144233 non-null  object 
 12  environment_freq         144233 non-null  float64
 13  environment_risk         144233 non-null  float64
 14  time

In [128]:
df['environment_risk'] = df['environment_risk'].astype(object)

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude='number').columns.tolist()

df[num_cols] = df[num_cols].astype(float)
df[num_cols] = df[num_cols].fillna(0)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(object)

df[cat_cols] = df[cat_cols].fillna('missing')

In [129]:
df['is_fraud'] = df['is_fraud'].astype(object)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           590540 non-null  float64
 1   is_fraud                 590540 non-null  object 
 2   transaction_time         590540 non-null  float64
 3   transaction_amount       590540 non-null  float64
 4   card_network             590540 non-null  object 
 5   card_type                590540 non-null  object 
 6   purchaser_email_domain   590540 non-null  object 
 7   device_type              590540 non-null  object 
 8   is_identity_seen_before  590540 non-null  object 
 9   user_os                  590540 non-null  object 
 10  user_browser             590540 non-null  object 
 11  environment              590540 non-null  object 
 12  environment_freq         590540 non-null  float64
 13  environment_risk         590540 non-null  object 
 14  time

In [130]:
df['device_type'].unique()

array(['missing', 'mobile', 'desktop'], dtype=object)

In [131]:
df['user_os'].unique()

array(['missing', 'Android', 'iOS', 'Other', 'macOS', 'Windows', 'Linux'],
      dtype=object)

In [132]:
df['device_typ_os'] = df['device_type'] + '_' + df['user_os']

In [133]:
df['device_typ_os'].unique()

array(['missing_missing', 'mobile_Android', 'mobile_iOS', 'desktop_Other',
       'desktop_macOS', 'desktop_Windows', 'missing_Other',
       'desktop_Linux', 'mobile_Other', 'mobile_macOS', 'desktop_Android',
       'mobile_Linux', 'mobile_Windows', 'desktop_iOS'], dtype=object)

In [134]:
ct = pd.crosstab(df['device_typ_os'], df['is_fraud'])
(pd.crosstab(df['device_typ_os'], df['is_fraud'], normalize='index')) * 100

is_fraud,0.0,1.0
device_typ_os,,
desktop_Android,100.000000,0.000000
desktop_Linux,92.741935,7.258065
desktop_Other,88.424733,11.575267
desktop_Windows,96.547029,3.452971
desktop_iOS,100.000000,0.000000
desktop_macOS,97.804303,2.195697
missing_Other,96.874087,3.125913
missing_missing,97.906150,2.093850
mobile_Android,91.399556,8.600444


In [135]:
chi2, p, dof, exp = chi2_contingency(ct)
print(p)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(cramers_v)

0.0
0.17594492676742396


In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           590540 non-null  float64
 1   is_fraud                 590540 non-null  object 
 2   transaction_time         590540 non-null  float64
 3   transaction_amount       590540 non-null  float64
 4   card_network             590540 non-null  object 
 5   card_type                590540 non-null  object 
 6   purchaser_email_domain   590540 non-null  object 
 7   device_type              590540 non-null  object 
 8   is_identity_seen_before  590540 non-null  object 
 9   user_os                  590540 non-null  object 
 10  user_browser             590540 non-null  object 
 11  environment              590540 non-null  object 
 12  environment_freq         590540 non-null  float64
 13  environment_risk         590540 non-null  object 
 14  time

In [137]:
df['known_env_risk'] = df['environment_risk'].astype(str) + '_' + df['is_identity_seen_before'].astype(str)

In [138]:
ct = pd.crosstab(df['known_env_risk'], df['is_fraud'])
((pd.crosstab(df['known_env_risk'], df['is_fraud'], normalize='index')) * 100).round(3)

is_fraud,0.0,1.0
known_env_risk,,
0.0_Found,87.301,12.699
0.0_New,94.594,5.406
0.0_missing,90.330,9.670
1.0_Found,94.491,5.509
1.0_New,96.933,3.067
1.0_missing,96.570,3.430
2.0_Found,80.628,19.372
2.0_New,90.664,9.336
2.0_missing,69.259,30.741


In [139]:
chi2, p, dof, exp = chi2_contingency(ct)
print(p)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(cramers_v)

0.0
0.17123194300282718


In [140]:
df['time_diff_log'].describe()

count    590540.000000
mean          2.610262
std           1.160858
min           0.000000
25%           1.791759
50%           2.639057
75%           3.401197
max           8.328209
Name: time_diff_log, dtype: float64

In [141]:
df['time_diff_bins'] = pd.qcut(df['time_diff_log'], q = 10 , labels=False) 

In [142]:
# import warnings
# warnings.filterwarnings('ignore')

# # ---------- 1. Define your categorical columns (all treated as categorical) ----------
# # Use your final DataFrame after preprocessing (no missing values, consistent types)
# # The columns you provided, with types adjusted:
# categorical_cols = [
#     'is_fraud',               # target
#     'card_network',
#     'card_type',
#     'purchaser_email_domain',
#     'device_type',
#     'is_identity_seen_before',
#     'user_os',
#     'user_browser',
#     'environment_risk',       # already object (categorical)
#     'time_hour',              # you want to treat as categorical
#     'amt_bins',               # binned amount
#     'day_of_week',            # already numeric but few categories
#     'device_typ_os',          # engineered feature
#     'known_env_risk',
#     'time_diff_bins'          # engineered feature
# ]

# # Ensure all are of type 'object' or 'category' for consistent handling
# df[categorical_cols] = df[categorical_cols].astype('object')

# # ---------- 2. Fill any remaining missing values with 'missing' (though you said none) ----------
# df_filled = df[categorical_cols].fillna('missing')

# # ---------- 3. Function to compute Cramér's V ----------
# def cramers_v(confusion_matrix):
#     """Calculate Cramér's V from a contingency table."""
#     chi2 = chi2_contingency(confusion_matrix)[0]
#     n = confusion_matrix.sum().sum()
#     phi2 = chi2 / n
#     r, k = confusion_matrix.shape
#     phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
#     rcorr = r - ((r-1)**2)/(n-1)
#     kcorr = k - ((k-1)**2)/(n-1)
#     return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# # ---------- 4. Compute all pairwise associations ----------
# # We'll store results in a dictionary of dataframes, one per feature
# results_per_feature = {}

# for i, feat1 in enumerate(categorical_cols):
#     records = []
#     for j, feat2 in enumerate(categorical_cols):
#         if i == j:
#             continue  # skip self
#         # Create contingency table
#         crosstab = pd.crosstab(df_filled[feat1], df_filled[feat2])
#         try:
#             chi2, p, dof, expected = chi2_contingency(crosstab)
#             v = cramers_v(crosstab)
#         except:
#             chi2, p, v = np.nan, np.nan, np.nan
        
#         records.append({
#             'feature_2': feat2,
#             'cramers_v': v,
#             'p_value': p,
#             'chi2': chi2
#         })
    
#     # Convert to DataFrame and sort by Cramér's V descending
#     df_pair = pd.DataFrame(records)
#     df_pair = df_pair.sort_values('cramers_v', ascending=False).reset_index(drop=True)
#     results_per_feature[feat1] = df_pair

# # ---------- 5. Display results per feature ----------
# for feat, df_res in results_per_feature.items():
#     print(f"\n{'='*80}")
#     print(f" Associations for feature: {feat}")
#     print('='*80)
#     print(df_res.to_string(index=False))
#     print()  # blank line for readability

In [152]:
df2 = df.drop(columns=['is_fraud', 'transaction_id', 'transaction_time', 
                       'time_day', 'time_diff', 'time_diff_min', 
                       'amt_log', 'new_day', 'day_id', 'day_of_week', 'transaction_amount', 'environment', 'environment_freq', 'time_diff_log'])
for i in df2.columns:
    '___________*************************_________'
    print(i)
    print(df[i].unique())

    '___________***************_________'

card_network
['discover' 'mastercard' 'visa' 'american express' 'missing']
card_type
['credit' 'debit' 'missing' 'debit or credit' 'charge card']
purchaser_email_domain
['missing' 'gmail.com' 'outlook.com' 'yahoo.com' 'mail.com'
 'anonymous.com' 'hotmail.com' 'verizon.net' 'aol.com' 'me.com'
 'comcast.net' 'optonline.net' 'cox.net' 'charter.net' 'rocketmail.com'
 'prodigy.net.mx' 'embarqmail.com' 'icloud.com' 'live.com.mx' 'gmail'
 'live.com' 'att.net' 'juno.com' 'ymail.com' 'sbcglobal.net'
 'bellsouth.net' 'msn.com' 'q.com' 'yahoo.com.mx' 'centurylink.net'
 'servicios-ta.com' 'earthlink.net' 'hotmail.es' 'cfl.rr.com'
 'roadrunner.com' 'netzero.net' 'gmx.de' 'suddenlink.net'
 'frontiernet.net' 'windstream.net' 'frontier.com' 'outlook.es' 'mac.com'
 'netzero.com' 'aim.com' 'web.de' 'twc.com' 'cableone.net' 'yahoo.fr'
 'yahoo.de' 'yahoo.es' 'sc.rr.com' 'ptd.net' 'live.fr' 'yahoo.co.uk'
 'hotmail.fr' 'hotmail.de' 'hotmail.co.uk' 'protonmail.com' 'yahoo.co.jp']
device_type
['missing' 'mobi

In [149]:
# Define the combinations (feature1, feature2) and a short name prefix
combinations = [
    ('card_network', 'user_os', 'card_net_user_os'),
    ('card_network', 'is_identity_seen_before', 'card_net_id_seen'),
    ('card_type', 'user_os', 'card_type_user_os'),
    ('card_type', 'is_identity_seen_before', 'card_type_id_seen'),
    ('purchaser_email_domain', 'device_type', 'pemail_dev_type'),
    ('purchaser_email_domain', 'is_identity_seen_before', 'pemail_id_seen'),
    ('purchaser_email_domain', 'environment_risk', 'pemail_env_risk'),
    ('purchaser_email_domain', 'user_os', 'pemail_user_os'),
    ('is_identity_seen_before', 'user_browser', 'id_seen_user_browser'),
    ('user_os', 'user_browser', 'user_os_browser'),
    ('user_browser', 'environment_risk', 'browser_env_risk'),
    ('user_browser', 'card_type', 'browser_card_type'),
    ('time_diff_bins', 'time_hour', 'time_diff_bins_hour'),
    ('is_identity_seen_before', 'amt_bins', 'id_seen_amt_bin'),
]

# Convert numeric-like columns to string to avoid errors during concatenation
# (time_diff_bins and amt_bins may be numeric; time_hour is numeric but we treat as category)
for col1, col2, _ in combinations:
    if col1 in df.columns and df[col1].dtype in ['int64', 'float64']:
        df[col1] = df[col1].astype(str)
    if col2 in df.columns and df[col2].dtype in ['int64', 'float64']:
        df[col2] = df[col2].astype(str)

# Create the combined features
for col1, col2, new_name in combinations:
    df[new_name] = df[col1].astype(str) + '_' + df[col2].astype(str)

# The new features are categorical. You can later apply label encoding or target encoding.

In [150]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 41 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           590540 non-null  float64
 1   is_fraud                 590540 non-null  object 
 2   transaction_time         590540 non-null  float64
 3   transaction_amount       590540 non-null  float64
 4   card_network             590540 non-null  object 
 5   card_type                590540 non-null  object 
 6   purchaser_email_domain   590540 non-null  object 
 7   device_type              590540 non-null  object 
 8   is_identity_seen_before  590540 non-null  object 
 9   user_os                  590540 non-null  object 
 10  user_browser             590540 non-null  object 
 11  environment              590540 non-null  object 
 12  environment_freq         590540 non-null  float64
 13  environment_risk         590540 non-null  object 
 14  time

In [153]:
for i in df2.columns:
    '___________*************************_________'
    print(i)
    print(len(df[i].unique()))

    '___________***************_________'

card_network
5
card_type
5
purchaser_email_domain
60
device_type
3
is_identity_seen_before
3
user_os
7
user_browser
10
environment_risk
4
time_hour
24
amt_bins
10
device_typ_os
14
known_env_risk
10
time_diff_bins
10
card_net_user_os
34
card_net_id_seen
15
card_type_user_os
24
card_type_id_seen
13
pemail_dev_type
177
pemail_id_seen
178
pemail_env_risk
213
pemail_user_os
315
id_seen_user_browser
28
user_os_browser
37
browser_env_risk
19
browser_card_type
34
time_diff_bins_hour
240
id_seen_amt_bin
30
